# DataMate — Qwen3.5-9B trên tập train (Kaggle T4 ×2), một notebook duy nhất
**Session options:** Accelerator **GPU T4 ×2**, **Internet On**. **Add Input:** dataset có thư mục **train** (có `answers.jsonl`).

- **Lần đầu:** notebook tự lưu Qwen3.5-9B phần text đã lượng tử hoá NF4 (~7 GB) và các wheel còn thiếu vào `/kaggle/working`, rồi chạy luôn. Dùng **Save Version → Save & Run All (Commit)** để output được giữ lại.
- **Các lần sau:** **Add Input → Your Work → Notebooks →** chính notebook này (chọn version có thư mục `qwen3.5-9b-nf4`). Notebook tự tìm model + wheel đã lưu, bỏ qua toàn bộ bước tải/lượng tử hoá/cài qua mạng.

Chỉ cần sửa cell cấu hình. Chạy `LIMIT = 1` trước để xem tốc độ, rồi tăng dần (0 = toàn bộ).

In [ ]:
LIMIT = 1  # số câu; 0 = toàn bộ
SEED = 42
CONTEXT_TOKENS = 32768  # input + output mỗi bước
TEMPERATURE = 0.7  # preset non-thinking của Qwen; 0 = greedy (dễ lặp)
HARNESS = "h4"  # đổi khi đổi harness để không ghi đè/so lẫn kết quả cũ
RESUME = False
FAST_KERNELS = True  # thử flash-linear-attention; chỉ giữ nếu kết quả khớp và nhanh hơn trên GPU này
MODEL_ID = "Qwen/Qwen3.5-9B"  # chỉ dùng khi phải tự chuẩn bị model
REPO_URL = "https://github.com/KhoaNotFound/ZAIC2026-BackpropEverytTime.git"
DATA_DIR = ""   # để trống: tự tìm thư mục train trong /kaggle/input
MODEL_DIR = ""  # để trống: model đã chuẩn bị trong /kaggle/input, nếu không có thì tự chuẩn bị

import json, os, shutil, subprocess, sys
from pathlib import Path

def find_dirs(predicate, root="/kaggle/input"):
    found = []
    for d, subdirs, files in os.walk(root):
        if predicate(set(files), set(subdirs)):
            found.append(d)
            subdirs.clear()
    return sorted(found)

if not DATA_DIR:
    found = find_dirs(lambda f, s: {"questions.jsonl", "answers.jsonl", "rulebook.md"} <= f and "data" in s)
    assert len(found) == 1, f"Cần đúng một thư mục train, tìm thấy {found}: gán DATA_DIR"
    DATA_DIR = found[0]
if not MODEL_DIR:
    found = find_dirs(lambda f, s: "zaic_model.json" in f)
    MODEL_DIR = found[0] if found else "/kaggle/working/qwen3.5-9b-nf4"
NEED_PREPARE = not Path(MODEL_DIR, "zaic_model.json").is_file()
found = find_dirs(lambda f, s: "packages.json" in f)
WHEELS_DIR = found[0] if found else "/kaggle/working/wheels"
OUTPUT = (f"/kaggle/working/{Path(MODEL_DIR).name}_{HARNESS}_train_limit{LIMIT}_seed{SEED}"
          f"_ctx{CONTEXT_TOKENS}_t{TEMPERATURE}/predictions.jsonl")
REPO = "/kaggle/working/zAIC"
print("DATA_DIR  =", DATA_DIR)
print("MODEL_DIR =", MODEL_DIR, "(sẽ chuẩn bị trong session này)" if NEED_PREPARE else "(đã chuẩn bị)")
print("WHEELS    =", WHEELS_DIR)
print("OUTPUT    =", OUTPUT)

In [ ]:
# Code + thư viện. Inference chạy trong subprocess nên cài xong không cần restart kernel.
if not Path(REPO).exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
os.chdir(REPO)
if not Path(WHEELS_DIR, "packages.json").is_file():
    # Tải đúng các gói image này còn thiếu (bỏ qua torch/CUDA), giữ lại cho các lần sau.
    subprocess.run([sys.executable, "scripts/download_wheels.py", "--output", WHEELS_DIR], check=True)
wheels = [str(p) for p in Path(WHEELS_DIR).iterdir() if p.suffix in (".whl", ".gz", ".zip")]
if wheels:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-index", "--no-deps", *wheels], check=True)
check = "import transformers, accelerate, bitsandbytes, duckdb, openpyxl, pandas; print(transformers.__version__)"
probe = subprocess.run([sys.executable, "-c", check], capture_output=True, text=True)
if probe.returncode or tuple(map(int, probe.stdout.split(".")[:2])) < (5, 13):
    print("Wheel không khớp image: cài qua mạng")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements.txt"], check=True)
import torch
print("transformers", subprocess.run([sys.executable, "-c", check], capture_output=True, text=True).stdout.strip(),
      "| PyTorch", torch.__version__, "| GPUs:",
      [f"{torch.cuda.get_device_name(i)} {torch.cuda.get_device_properties(i).total_memory / 2**30:.1f} GiB"
       for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 2, "Cần 2 GPU: chọn Accelerator GPU T4 x2"

In [ ]:
# Kernel nhanh cho các layer linear-attention của Qwen3.5 (mặc định transformers dùng bản PyTorch chậm).
# So với bản PyTorch trên một model nhỏ ngẫu nhiên: chỉ giữ nếu logits khớp VÀ nhanh hơn, không thì gỡ ra.
FLA = ["flash-linear-attention", "fla-core"]
def kernel_check(*args):
    r = subprocess.run([sys.executable, "scripts/check_fast_kernels.py", *args], capture_output=True, text=True)
    lines = r.stdout.strip().splitlines()
    return json.loads(lines[-1]) if lines else {"ok": False, "error": r.stderr[-500:]}
if FAST_KERNELS:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "--quiet", *FLA], capture_output=True)
    ref = kernel_check("--save", "/tmp/kernel_ref.pt")
    fast_dir = Path(WHEELS_DIR, "fast")
    if not any(fast_dir.glob("*.whl")):
        fast_dir = Path("/kaggle/working/wheels/fast")
        subprocess.run([sys.executable, "-m", "pip", "download", "--quiet", "--no-deps", "-d", str(fast_dir),
                        *FLA, "einops"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-index", "--no-deps",
                    *map(str, fast_dir.glob("*.whl"))], check=True)
    result = kernel_check("--compare", "/tmp/kernel_ref.pt")
    print("PyTorch:", ref)
    print("flash-linear-attention:", result)
    if not result.get("ok"):
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "--quiet", *FLA], check=True)
        print("=> Không dùng kernel nhanh (sai lệch, lỗi hoặc không nhanh hơn).")
    else:
        print(f"=> Dùng flash-linear-attention, nhanh hơn {result['speedup']:.1f}x trên phép thử.")

In [ ]:
# Chỉ chạy lần đầu: lưu model NF4 phần text (~10-15 phút), rồi xoá bản tải gốc ~19 GB khỏi đĩa.
if NEED_PREPARE:
    subprocess.run([sys.executable, "scripts/prepare_model.py", "--model", MODEL_ID,
                    "--quantization", "4bit", "--output", MODEL_DIR], check=True)
    shutil.rmtree(Path.home() / ".cache/huggingface/hub", ignore_errors=True)
print(Path(MODEL_DIR, "zaic_model.json").read_text())

In [ ]:
env = dict(os.environ, HF_HUB_OFFLINE="1")  # model local: không gọi Hugging Face
command = [sys.executable, "inference.py", "--data-dir", DATA_DIR, "--limit", str(LIMIT), "--seed", str(SEED),
           "--output", OUTPUT, "--model", MODEL_DIR, "--device-map", "split",
           "--context-tokens", str(CONTEXT_TOKENS), "--temperature", str(TEMPERATURE), "--evaluate"]
if RESUME:
    command.append("--resume")
subprocess.run(command + ["--dry-run"], check=True, env=env)  # kiểm tra dữ liệu, không load model
subprocess.run(command, check=True, env=env)  # chạy rồi chấm điểm ngay

In [ ]:
out = Path(OUTPUT)
for name in ["summary", "evaluation"]:
    path = out.with_suffix(f".{name}.json")
    if path.exists():
        report = json.loads(path.read_text())
        report.pop("details", None)  # từng câu: xem predictions.evaluation.json
        print(name, json.dumps(report, ensure_ascii=False, indent=1)[:4000])
print("Artifacts:", sorted(p.name for p in out.parent.glob("predictions.*")))

In [ ]:
# Trace gọn từng câu: model trả lời gì mỗi bước và harness phản hồi lỗi gì (gửi đoạn này khi cần chẩn đoán).
traces = Path(OUTPUT).with_suffix(".traces.jsonl")
for line in traces.read_text().splitlines() if traces.exists() else []:
    t = json.loads(line)
    print(f"== {t['id']} {t['status']} {t.get('error') or ''}")
    for s in t["trace"]:
        usage = s.get("usage") or {}
        print(f"  [{s['step']}] in={usage.get('input_tokens')} cached={usage.get('cached_tokens')} "
              f"out={usage.get('output_tokens')} {usage.get('generation_seconds', 0):.1f}s")
        print("     reply:", (s.get("response") or "")[:400].replace("\n", " "))
        if s.get("tool_error"):
            print("     error:", s["tool_error"][:300].replace("\n", " "))

Đổi số câu: sửa `LIMIT`, chạy lại từ cell cấu hình (thư mục output tự đổi theo model/harness/LIMIT/SEED). Chạy tiếp một đợt bị ngắt: giữ nguyên cấu hình, đặt `RESUME = True`.
Nếu dừng do lỗi (ví dụ OOM), xem trace ở cell trên; restart kernel trước khi chạy lại. Nghi prefix cache gây lỗi: thêm `"--no-prefix-cache"` vào `command`. Nghi kernel nhanh gây lỗi: đặt `FAST_KERNELS = False`.